In [2]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# link do repozytorium: https://www.kaggle.com/datasets/kazanova/sentiment140

#dodaj właściwe kolumny zgodnie z dokumentacją oraz zdefiniuj plik z danymi w ramce pandas
columns = ["target","ids","date","flag","user","text"]
data = pd.read_csv("twitter_16.csv", names=columns, encoding='latin1')

# Print the shape of dataframe
print(data.shape)

# Print top 5 rows
data.head(5)

(1600000, 6)


,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [3]:
# Przekształć etykiety 'target' na wartości binarne (0 dla negatywnych, 1 dla pozytywnych)
data['target'] = data['target'].apply(lambda x: 1 if x == 4 else 0)

# Podziel na zbiory treningowy i testowy
train_size = int(len(data) * 0.8)
train_data = data[:train_size]
test_data = data[train_size:]

# Tokenizacja tekstu
max_words = 10000
maxlen = 100  # Długość tweetów

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_data['text'])

# Zamień tekst na sekwencje liczbowe
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data['text']), maxlen=maxlen)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data['text']), maxlen=maxlen)
y_train = train_data['target'].values
y_test = test_data['target'].values

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=maxlen),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')  # Wyjście binarne
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
batch_size = 500
epochs = 5

history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test),
    verbose=1
)


Epoch 1/5
 251/2560 ━━━━━━━━━━━━━━━━━━━━ 12:56 336ms/step - accuracy: 0.8185 - loss: 0.3994

In [ ]:
score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print(f"Test Accuracy: {acc}")


In [ ]:
def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=maxlen)
    prediction = model.predict(padded)
    return "Positive" if prediction > 0.5 else "Negative"

# Przykład prognozy
print(predict_sentiment("I love this product!"))
